Classes bundle **data** (attributes) with the **behaviour** that acts on it (methods). In scientific code they are ideal for things like a *simulation case*, a *mesh*, or a *material model* — a coherent object you pass around and extend.

::: {.callout-tip}
## Why learn this?
When a script grows past a couple hundred lines, passing a dozen loose variables around becomes a nightmare. Bundling the data with the behaviour that acts on it — a `Case`, a `Signal`, a `Mesh` — keeps large code understandable.

- **Imagine you need** to carry a simulation's velocity, density and grid together and ask it for its Reynolds number — an *object* holds all of that and answers with one method call.
- **Imagine you need** a `ChannelCase` and a `JetCase` that share 90% of their code — *inheritance* lets you write the shared part once.
:::

## A first class
`__init__` sets up each instance; `self` is the instance the method is called on. `__repr__` gives a useful printed form.

In [1]:
class Fluid:
    """A simple incompressible fluid."""

    def __init__(self, name, rho, mu):
        self.name = name
        self.rho = rho          # density  [kg/m^3]
        self.mu = mu            # dyn. viscosity [Pa s]

    def reynolds(self, u, length):
        """Reynolds number for velocity u and length scale."""
        return self.rho * u * length / self.mu

    def __repr__(self):
        return f"Fluid(name={self.name!r}, rho={self.rho}, mu={self.mu})"

water = Fluid('water', 1000.0, 1.0e-3)
print(water)
print('Re =', water.reynolds(u=1.0, length=0.05))

Fluid(name='water', rho=1000.0, mu=0.001)
Re = 50000.0


## Properties and encapsulation
A `@property` exposes a *computed* value as if it were an attribute, and lets you validate on assignment — without changing how callers use the object.

In [2]:
class Pipe:
    def __init__(self, diameter):
        self.diameter = diameter        # goes through the setter below

    @property
    def diameter(self):
        return self._d

    @diameter.setter
    def diameter(self, value):
        if value <= 0:
            raise ValueError("diameter must be positive")
        self._d = value

    @property
    def area(self):
        """Cross-sectional area — computed, read-only."""
        import math
        return math.pi * (self._d / 2) ** 2

pipe = Pipe(0.05)
print('area =', pipe.area)
try:
    pipe.diameter = -1
except ValueError as e:
    print('rejected:', e)

area = 0.001963495408493621
rejected: diameter must be positive


**Imagine you need** a `ChannelCase`, a `PipeCase` and a `JetCase` that differ only in their geometry but share all the post-processing. This is super easy in Python using *inheritance* from a common base class.

## Inheritance
A subclass reuses and extends a base class; `super()` calls the parent's implementation.

In [3]:
class NonNewtonianFluid(Fluid):
    """Adds a power-law index n on top of a Fluid."""

    def __init__(self, name, rho, mu, n):
        super().__init__(name, rho, mu)   # reuse Fluid.__init__
        self.n = n

    def __repr__(self):
        base = super().__repr__()[:-1]     # drop trailing ')'
        return f'{base}, n={self.n})'

blood = NonNewtonianFluid('blood', 1060.0, 3.5e-3, 0.7)
print(blood)
print('inherited method Re =', round(blood.reynolds(0.2, 0.004), 2))
print('is a Fluid?', isinstance(blood, Fluid))

Fluid(name='blood', rho=1060.0, mu=0.0035, n=0.7)
inherited method Re = 242.29
is a Fluid? True


## Dunder methods make objects feel native
Implementing special (`__dunder__`) methods lets your objects work with `len()`, `==`, `[]`, arithmetic, and so on.

In [4]:
class Vector:
    def __init__(self, *components):
        self.c = tuple(components)
    def __len__(self):
        return len(self.c)
    def __getitem__(self, i):
        return self.c[i]
    def __add__(self, other):
        return Vector(*(a + b for a, b in zip(self.c, other.c)))
    def __eq__(self, other):
        return self.c == other.c
    def __repr__(self):
        return f'Vector{self.c}'

v = Vector(1, 2, 3)
w = Vector(4, 5, 6)
print(v + w, '| len =', len(v), '| v[0] =', v[0], '| v==v:', v == Vector(1, 2, 3))

Vector(5, 7, 9) | len = 3 | v[0] = 1 | v==v: True


**Imagine you need** a config object with a dozen fields and a readable printout. This is super easy in Python using `@dataclass`, which writes the boilerplate for you.

## `dataclass` — classes without boilerplate
For plain data containers, `@dataclass` writes `__init__`, `__repr__`, and `__eq__` for you.

In [5]:
from dataclasses import dataclass, field

@dataclass
class SimCase:
    name: str
    reynolds: float
    mesh_cells: int = 10_000
    tags: list = field(default_factory=list)

case = SimCase('channel', 5000, tags=['LES'])
print(case)
print('equal?', case == SimCase('channel', 5000, tags=['LES']))

SimCase(name='channel', reynolds=5000, mesh_cells=10000, tags=['LES'])
equal? True


## Self-tests

**1.** Give `Fluid` a `kinematic_viscosity` **property** returning $\nu=\mu/\rho$.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
class Fluid2(Fluid):
    @property
    def kinematic_viscosity(self):
        return self.mu / self.rho

print(Fluid2('water', 1000.0, 1.0e-3).kinematic_viscosity)

**2.** Make a `@dataclass` `Airfoil` with fields `name: str`, `chord: float`, `alpha: float = 0.0`.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
from dataclasses import dataclass
@dataclass
class Airfoil:
    name: str
    chord: float
    alpha: float = 0.0
print(Airfoil('NACA0012', 0.1))

## Turbulence in practice: a `Signal` class

Wrap a velocity record in a class whose methods are the standard Reynolds operations — mean, fluctuation, and rms.

In [8]:
import numpy as np

class Signal:
    """A time series of one velocity component."""
    def __init__(self, t, u):
        self.t = np.asarray(t)
        self.u = np.asarray(u)
    @property
    def mean(self):
        return self.u.mean()
    def fluctuations(self):
        return self.u - self.mean
    def rms(self):
        return float(np.sqrt((self.fluctuations() ** 2).mean()))
    def __len__(self):
        return self.u.size

t = np.linspace(0, 1, 500)
s = Signal(t, 10 + np.sin(2 * np.pi * 5 * t))
print(f'len={len(s)}, <u>={s.mean:.3f}, u_rms={s.rms():.4f}')

len=500, <u>=10.000, u_rms=0.7064


**Self-test.** Subclass `Signal` and add an `intensity` **property** returning the turbulence intensity $I = u_{\mathrm{rms}}/\langle u\rangle$.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
class Signal2(Signal):
    @property
    def intensity(self):
        return self.rms() / self.mean

print(Signal2(t, 10 + np.sin(2 * np.pi * 5 * t)).intensity)